🌐 [English](dx_rt_01_beginner.ipynb) | [한국어](dx_rt_01_beginner.ko.ipynb) | **日本語** | [中文](dx_rt_01_beginner.zh.ipynb)

# DEEPX Tutorial 06-1 - DX-RT 入門

このノートブックでは、コンパイル済みの DXNN モデルを DEEPX NPU 上で実行するランタイム層を紹介します。

## 学習目標

このチュートリアルを終えると、次のことができるようになります。

- DX-RT が何を行い、何がアプリケーション側の責任として残るかを説明する
- インストール済みのランタイムツールと NPU の状態を確認する
- DXNN モデルの入力、出力、タスクグラフを調べる
- 単一実行、最大スループット、目標 FPS の各モードを区別する
- `dxrun` で制御されたダミー入力のベンチマークを実行する
- CPU タスクを含むモデルについて、NPU のみの実行と ORT 有効時の実行を比較する
- 検査、推論、監視それぞれに適した CLI ツールを選ぶ

このチュートリアルは SDK のソースツリーを変更しません。チュートリアルで生成される成果物とログはすべて `<dx-tutorials>/notebooks/T06-DX-Runtime/workspace` 配下に保存されます。

## コースマップ

| ノートブック | 主な内容 |
|---|---|
| 入門 (Beginner) | DX-RT の役割、デバイスの確認、DXNN の検査、CLI による推論 |
| 中級 (Intermediate) | Python と C++ の API、同期/非同期実行、バッチ、バッファ |
| 上級 (Advanced) | リソースバインディング、プロファイリング、監視、複数入力/メモリからのロード、リリース検証 |

DXNN モデルのコントラクトと DX-RT CLI をすでに理解している場合を除き、ノートブックは順番に進めてください。

## 前提条件

- DX-RT がインストール済み(チュートリアル 01 第 3 節)で、DEEPX NPU が `/dev/dxrt0` として認識されていること。
- ResNet50 サンプルモデル。存在しない場合は、セットアップセルが `dx_app/setup.sh` でダウンロードします。
- 第 6 節で使用する YOLO26-S サンプルモデル(21 MB)。存在しない場合は、その節で同じ方法でダウンロードします。
- `sudo` は不要です。所要時間は約 10 分です。

次のセルは SDK の場所を特定し、これらの要件を確認してステータス表を表示します。`MISSING` と表示された項目には、それを提供する手順が添えられます。

## 1. DX-RT の位置づけ

DX-COM は `.dxnn` モデルを生成します。DX-RT はそのモデルを読み込み、推論バッファとジョブを管理し、デバイスドライバーと通信して、出力テンソルを返します。

<img src="assets/dx-rt-runtime-workflow.svg" style="max-width: 1100px; width: 100%;" alt="DX-RT 推論ワークフロー">

| DX-RT が担当するもの | アプリケーションが担当するもの |
|---|---|
| DXNN の読み込みと検証 | 入力の取得 |
| デバイスの選択と NPU のスケジューリング | モデル固有の前処理 |
| ランタイムの入出力バッファ | モデル固有の後処理 |
| 同期ジョブと非同期ジョブ | 製品としての動作と可視化 |
| ランタイムのプロファイリングとデバイス照会 | エンドツーエンドの精度とサービスレベルの指標 |

> ランタイム呼び出しの成功は、モデルが実行されたことを示すだけです。前処理、デコード、アプリケーションの精度が正しいことを証明するものではありません。

## 2. チュートリアルワークスペースの初期化

セットアップセルは `config.json` から共有 SDK の場所を読み取ります。作成するのは、このチュートリアル内のローカルなリンクとディレクトリだけです。

```text
T06-DX-Runtime/
├── assets/
├── dx_rt_01_beginner.ipynb
├── dx_rt_02_intermediate.ipynb
├── dx_rt_03_advanced.ipynb
└── workspace/
    ├── models/       # links to SDK models
    ├── reports/      # dxparse and benchmark reports
    ├── profiler/     # profiler JSON and visualizations
    ├── cpp/          # generated C++ examples and build files
    ├── multi_input/  # link to the two-input model from Tutorial 05-3 (Advanced)
    └── .venv-dxrt/   # DX-RT Python binding environment (created in Intermediate section 2)
```

このチュートリアルでは、DX-APP のリソースセットアップでダウンロードされる `resnet50_224x224.dxnn` を使用します。存在しない場合、セットアップセルはそれを取得するための正確なターミナルコマンドを表示します。

In [ ]:
# Load the shared DEEPX tutorial paths (identical first code cell in every notebook).
from pathlib import Path; import importlib, sys
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial_paths.py").is_file())))
import tutorial_paths; importlib.reload(tutorial_paths)  # pick up updates without restarting the kernel
from tutorial_paths import setup_tutorial
import os
import sys

paths = setup_tutorial(needs=["dx_rt", "npu"])
paths.require("dx_rt", "npu")

MODEL_DIR = WORKSPACE_DIR / "models"
REPORT_DIR = WORKSPACE_DIR / "reports"
PROFILER_DIR = WORKSPACE_DIR / "profiler"
CPP_DIR = WORKSPACE_DIR / "cpp"

for path in (WORKSPACE_DIR, MODEL_DIR, REPORT_DIR, PROFILER_DIR, CPP_DIR):
    path.mkdir(parents=True, exist_ok=True)

MODEL_SOURCE = DX_WORKSPACE_DIR / "res" / "models" / "resnet50_224x224.dxnn"
if not MODEL_SOURCE.is_file():
    # Download the sample model with the DX-APP setup script (same command as in a terminal).
    !cd "{DX_APP_DIR}" && bash setup.sh --models resnet50 --no-force

MODEL_PATH = MODEL_DIR / MODEL_SOURCE.name
if MODEL_PATH.is_symlink() and not MODEL_PATH.exists():
    MODEL_PATH.unlink()   # stale link from a previous SDK location
if not MODEL_PATH.exists():
    MODEL_PATH.symlink_to(MODEL_SOURCE)

os.chdir(WORKSPACE_DIR)
print(f"T06 workspace : {WORKSPACE_DIR}")
print(f"Model         : {MODEL_PATH} -> {MODEL_SOURCE}")


### 2.1 ノートブックの外で使用するコマンド

上に表示されたモデルダウンロードコマンドは、通常のシェルコマンドです。

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
bash setup.sh --models resnet50 --no-force
```

`--no-force` は、すでにダウンロード済みのリソースをそのまま保持します。その後チュートリアルは自身の `workspace/models` ディレクトリ配下にシンボリックリンクを作成します。SDK のモデルをコピーしたり変更したりはしません。

## 3. ランタイムのインストールを確認する

以下のコマンドは、ターミナルに入力するコマンドとまったく同じものです。出力ファイルは作成しません。

In [ ]:
!command -v dxrun
!command -v dxparse
!command -v dxcli
!command -v dxtop
!command -v dxbenchmark
!dxcli --version


### 3.1 デバイスの確認

`dxcli --status` は、デバイスを明示的に選択しない限り、利用可能なすべてのアクセラレーターを照会します。

```bash
dxcli --status
dxcli -s          # short form of the same command
```

正常な結果では少なくとも 1 台のデバイスが識別されるはずです。デバイスが表示されない場合はここで止まり、モデルをテストする前にドライバー、ファームウェア、物理的な接続、インストール状態を確認してください。

In [ ]:
!dxcli --status


## 4. DXNN モデルのコントラクトを調べる

ランタイムアプリケーションは、コンパイル済みモデルと次の項目すべてについて一致している必要があります。

| コントラクト項目 | 重要な理由 |
|---|---|
| 入力テンソルの名前と数 | 複数入力モデルでは正しい対応付けが必要 |
| 形状 | アプリケーションは必要な要素数を確保しなければならない |
| データ型 | dtype の不一致はエラーや不正なデータの原因になる |
| レイアウト | NHWC と NCHW は同じ値を異なる順序で格納する |
| 前処理の境界 | 一部の前処理はすでに DXNN にコンパイル済みの場合がある |
| 出力テンソル | 後処理は正しい名前、形状、順序を使わなければならない |
| CPU タスク | ORT が有効なランタイムパスが必要 |

`dxparse` は推論を実行せずにこのコントラクトを読み取ります。`-v` を付けるとタスクの依存関係とメモリ情報が追加されます。標準的なシェルのリダイレクトでレポートを保存し、次のセルで使用します。`dxparse` はファイルに書き込む場合でも ANSI エスケープコードで出力に色を付けるため、`sed` フィルターでそれらを除去し、保存されたレポートをプレーンテキストに保ちます。

同等のターミナルコマンド:

```bash
dxparse -m <T06-DX-Runtime>/workspace/models/resnet50_224x224.dxnn -v \
  | sed 's/\x1b\[[0-9;]*m//g' \
  > <T06-DX-Runtime>/workspace/reports/resnet50_dxparse.txt
```

In [ ]:
DXPARSE_REPORT = REPORT_DIR / "resnet50_dxparse.txt"

!dxparse -m "{MODEL_PATH}" -v | sed 's/\x1b\[[0-9;]*m//g' > "{DXPARSE_REPORT}"


### 4.1 保存したコントラクトの確認

次のセルは、チュートリアルワークスペース内に作成されたレポートを読み込みます。モデルバージョン、コンパイラバージョン、入出力テンソル、タスクの種類、メモリサイズを確認してください。

In [ ]:
!sed -n '1,220p' "{DXPARSE_REPORT}"


## 5. `dxrun` のモードを理解する

| モード | オプション | 主な動作 | 用途 |
|---|---|---|---|
| 単一実行 | `--single` | 1 コアでの逐次的な単一入力推論 | 基本的な実行とレイテンシの確認 |
| ベンチマーク | `--benchmark` | 利用可能なランタイムパイプラインを常にビジー状態に保つ | 最大スループットの比較 |
| 目標 FPS | `--fps N` | 要求されたレートで処理を投入する | 製品相当の負荷での容量と安定性の確認 |

`--time` はテストの実行時間を制御し、`--loops` より優先されます。`--warmup-runs` は初期のウォームアップ実行を測定から除外します。結果を比較する際は、同じモデル、オプション、実行時間、システム状態を使用してください。

### 5.1 リクエストを 1 回実行する

同等のターミナルコマンド:

```bash
cd <T06-DX-Runtime>/workspace
dxrun -m models/resnet50_224x224.dxnn --single --loops 1 --verbose
```

In [ ]:
!cd "{WORKSPACE_DIR}" && dxrun -m "models/{MODEL_PATH.name}" --single --loops 1 --verbose


### 5.2 最大スループットを測定する

このベンチマークはダミー入力を使用します。ランタイム性能の測定には適していますが、分類精度の測定には適していません。

同等のターミナルコマンド:

```bash
cd <T06-DX-Runtime>/workspace
dxrun -m models/resnet50_224x224.dxnn \
      --benchmark \
      --time 5 \
      --warmup-runs 5 \
      --buffer-count 6
```

In [ ]:
!cd "{WORKSPACE_DIR}" && dxrun -m "models/{MODEL_PATH.name}"     --benchmark     --time 5     --warmup-runs 5     --buffer-count 6


### 5.3 目標ワークロードをテストする

目標 FPS での実行は、システムが要求された到着レートを維持できるかを問うものです。最大スループットのベンチマークとは異なります。

同等のターミナルコマンド:

```bash
cd <T06-DX-Runtime>/workspace
dxrun -m models/resnet50_224x224.dxnn --fps 30 --time 5 --warmup-runs 5
```

In [ ]:
!cd "{WORKSPACE_DIR}" && dxrun -m "models/{MODEL_PATH.name}"     --fps 30     --time 5     --warmup-runs 5


## 6. CPU タスクと `--use-ort`

DXNN グラフには NPU タスクと CPU タスクを含めることができます。`--use-ort` は、NPU で実行されない CPU 側のサブグラフに対して ONNX Runtime を有効にします。

- `--use-ort` をやみくもに追加しないでください。
- まず `dxparse -v` でタスクグラフを確認してください。
- 比較するすべての結果で同じ ORT 設定を使用してください。
- ランタイムが ORT サポートなしでビルドされている場合、CPU タスクの実行は利用できません。

ここまで使用してきた ResNet50 モデルは NPU のみのグラフなので、このオプションを付けても何も変わりません。Model Zoo の YOLO26-S は異なります。その検出ヘッドは CPU タスク(`npu_0 -> cpu_0`)で終わり、6 つの生のヘッドテンソルを最終的な `[1, 300, 6]` の検出結果にデコードします。次の 3 つのセルは、モデルが存在しない場合にダウンロードし、そのタスクグラフを表示し、`dxrun` を 2 回(NPU のみ、次に `--use-ort` 付き)実行します。

同等のターミナルコマンド:

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
bash setup.sh --models yolo26s --no-force        # only when the model is missing

cd <T06-DX-Runtime>/workspace
dxparse -m models/yolo26-s_640x640.dxnn -v
dxrun -m models/yolo26-s_640x640.dxnn --benchmark --time 5 --warmup-runs 5
dxrun -m models/yolo26-s_640x640.dxnn --benchmark --time 5 --warmup-runs 5 --use-ort
```

In [ ]:
YOLO26S_SOURCE = DX_WORKSPACE_DIR / "res" / "models" / "yolo26-s_640x640.dxnn"
if not YOLO26S_SOURCE.is_file():
    # Download the sample model with the DX-APP setup script (same command as in a terminal).
    !cd "{DX_APP_DIR}" && bash setup.sh --models yolo26s --no-force

YOLO26S_PATH = MODEL_DIR / YOLO26S_SOURCE.name
if YOLO26S_PATH.is_symlink() and not YOLO26S_PATH.exists():
    YOLO26S_PATH.unlink()   # stale link from a previous SDK location
if not YOLO26S_PATH.exists():
    YOLO26S_PATH.symlink_to(YOLO26S_SOURCE)
print(f"Model: {YOLO26S_PATH} -> {YOLO26S_SOURCE}")

# Task graph only: the NPU task feeds a CPU task, which produces the model output.
!dxparse -m "{YOLO26S_PATH}" -v | sed 's/\x1b\[[0-9;]*m//g' | sed -n '/Task Graph Information/,/Task Details/p'


In [ ]:
# 1. NPU only: the CPU task is skipped, so the "Outputs" are the six raw head tensors.
!cd "{WORKSPACE_DIR}" && dxrun -m "models/{YOLO26S_PATH.name}" --benchmark --time 5 --warmup-runs 5


In [ ]:
# 2. With ORT: Task[1] cpu_0 runs as well and the final output is output0 [1, 300, 6].
!cd "{WORKSPACE_DIR}" && dxrun -m "models/{YOLO26S_PATH.name}" --benchmark --time 5 --warmup-runs 5 --use-ort


**確認ポイント**

| | NPU のみ | `--use-ort` |
|---|---|---|
| `dxrun` が表示するタスク | `Task[0] npu_0` | `Task[0] npu_0` と `Task[1] cpu_0` |
| 出力 | 6 つの生のヘッドテンソル(`.../Conv_output_0`) | `output0 [1, 300, 6]`: デコード済みの検出結果 |
| FPS | 純粋な NPU スループット | CPU デコードを含むエンドツーエンドのスループット |

高速なデスクトップ CPU では、2 つの FPS 値は数パーセント以内に収まることがよくあります。DX-RT は NPU が次の入力を処理している間に CPU タスクをパイプラインで実行するためです。非力なホスト CPU では CPU タスクがボトルネックになり、FPS の差が広がります。常に異なるのは出力です。`--use-ort` なしでは、アプリケーションは 6 つの生テンソルを自分でデコードしなければなりません。CPU タスクを持つ検出モデルでは、アプリケーションで測定したスループットと比較すべき値は ORT を有効にした数値です。

## 7. CLI 名と後方互換性

現在のチュートリアルは新しいコマンド名を使用します。古いスクリプトは互換エイリアスを通じて引き続き動作します。

| 現在のコマンド | 以前のエイリアス | 目的 |
|---|---|---|
| `dxparse` | `parse_model` | DXNN モデルを検査する |
| `dxrun` | `run_model` | モデルを実行またはベンチマークする |
| `dxcli` | `dxrt-cli` | ランタイムのデバイスインターフェースを照会する |

ログやドキュメントの一貫性を保つため、新しいコードでは現在の名前を使用してください。

## 8. リアルタイム監視

`dxtop` は対話型で、ターミナルを継続的に再描画します。ノートブックのセルではなく、別の JupyterLab ターミナルで実行してください。

```bash
dxtop
```

**File → New → Terminal** を開き、コマンドを実行します。`q` を押すと終了します。別のターミナルで `dxrun` を実行しながら、使用率、NPU メモリ、温度、電圧、クロックを監視してください。

## 9. トラブルシューティング

| 症状 | 表示される内容 | 考えられる原因 | 対処 |
|---|---|---|---|
| コマンドが見つからない | `dxrun: command not found` | DX-RT がインストールされていないか、`/usr/local/bin` が `PATH` に含まれていない | チュートリアル 01 第 3 節を完了し、新しいターミナルを開く |
| デバイスがない | `dxcli --status` に `No device found` | ドライバー、ファームウェア、または物理的な接続 | `lsmod | grep dxrt` を確認し、再起動するか、ホストの電源を入れ直す |
| モデルが拒否される | `dxrun` のバージョンまたは形式のエラー | DXNN が互換性のない DX-COM でコンパイルされた | `dxparse` が表示するバージョンとインストール済みランタイムを比較し、必要なら再コンパイルする |
| CPU タスクのエラー | CPU タスクを持つモデルでの ORT エラー | ランタイムが ONNX Runtime なしでビルドされた | ORT が有効なビルドを使用し、`--use-ort` を付ける |
| 性能が想定外 | 実行ごとに FPS が変動する | ウォームアップなし、サーマルスロットリング、または他の負荷 | `--warmup-runs`、固定の実行時間、同一のオプション、アイドル状態のホストを使用する |

## 10. まとめ

### 10.1 完了したランタイムワークフロー

**ツールとデバイスの確認**  
→ **DXNN コントラクトの検査**  
→ **推論を 1 回実行**  
→ **最大スループットの測定**  
→ **目標ワークロードのテスト**  
→ **NPU のみと ORT 有効時の実行の比較**

<img src="assets/dx-rt-runtime-workflow.svg" style="max-width: 1000px; width: 100%;" alt="DX-RT 推論ワークフロー">

### 10.2 ツールダッシュボード

| 質問 | ツールまたはオプション | 根拠 |
|---|---|---|
| NPU は認識されているか? | `dxcli --status` | デバイスの状態 |
| モデルは何を期待しているか? | `dxparse -v` | テンソルとタスクのコントラクト |
| リクエストを 1 回実行できるか? | `dxrun --single` | 基本的な実行とレイテンシ |
| 最大スループットはどれくらいか? | `dxrun --benchmark` | 制御されたダミー入力での FPS |
| 30 FPS を維持できるか? | `dxrun --fps 30` | 目標負荷での動作 |
| モデルは CPU を必要とするか? | `dxparse -v`、`dxrun --use-ort` | タスクグラフとデコード済み出力 |
| 時間の経過とともに何が起きるか? | `dxtop` | デバイスのライブ状態 |

### 10.3 完了チェックリスト

- [ ] DX-RT とアプリケーションの境界を特定する
- [ ] インストール済みの CLI ツールを確認する
- [ ] NPU の状態を照会する
- [ ] 詳細な DXNN レポートを保存して確認する
- [ ] 単一実行、ベンチマーク、目標 FPS の各モードを実行する
- [ ] CPU タスクを持つモデルで NPU のみの実行と `--use-ort` 実行を比較する
- [ ] ランタイム性能とモデル精度を区別する
- [ ] 実際のアプリケーションデータで前処理、後処理、精度を検証する

> **覚えておくこと:** 性能を比較する際は、モデル、コマンド、ランタイムバージョン、デバイスの状態、ワークロードを正確に同じに保ってください。

### 10.4 次のステップ

**中級チュートリアル**に進み、同じランタイムの概念を Python と C++ の API で実装し、同期、非同期、バッチ、バッファ管理の選択肢を比較しましょう。